In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import matplotlib.pyplot as plt

from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

In [ ]:
DISEASE_CLASSES = [
    "Apical periodontitis",
    "Periodontitis",
    "Gingivitis",
    "Crooked teeth",
    "Damaged or missing teeth",
    "Impacted tooth",
    "Caries",
    "Pulpitis"
]

In [ ]:
class CBCTDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        fid = str(self.df.iloc[idx]["Filename"])

        img = torch.load(
            os.path.join("MMDental_preprocessed", f"{fid}.pt"),
            weights_only=True
        )

        label = torch.tensor(
            self.df.iloc[idx]["label"],
            dtype=torch.long
        )

        return img, label



In [ ]:
filtered_df = pd.read_csv("mmdental_multiclass_new.csv")
print(filtered_df["label"].value_counts())
print(filtered_df["Filename"].nunique())

label
5    65
4    54
0    47
2    45
7    39
6    37
1    27
3     8
Name: count, dtype: int64
322


In [ ]:
class_counts = filtered_df["label"].value_counts().sort_index().values
class_weights = 1.0 / torch.tensor(class_counts, dtype=torch.float32)
class_weights = class_weights / class_weights.sum() * len(DISEASE_CLASSES)
print(f"\nClass weights: {class_weights}")


Class weights: tensor([0.5827, 1.0143, 0.6086, 3.4234, 0.5072, 0.4213, 0.7402, 0.7022])


In [ ]:
dataset = CBCTDataset(filtered_df)

first_img, first_label = dataset[0]

# Print the size of the image tensor
print(first_img.size())

torch.Size([1, 224, 224, 224])


In [ ]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    filtered_df,
    test_size=0.3,
    random_state=42,
    stratify=filtered_df["label"]
)

In [ ]:
from torch.utils.data import DataLoader
train_loader = DataLoader(CBCTDataset(train_df), batch_size=1, shuffle=True, pin_memory=True)
val_loader   = DataLoader(CBCTDataset(val_df), batch_size=1, pin_memory=True)

In [ ]:
for batch_idx, (x, y) in enumerate(train_loader):
    print("Batch", batch_idx)
    break

Batch 0


In [ ]:
class ConvBlock3D(nn.Module):
    def __init__(self, in_ch, out_ch, dropout):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 3, padding=1),
            nn.InstanceNorm3d(out_ch),
            nn.LeakyReLU(inplace=True),
            nn.Conv3d(out_ch, out_ch, 3, padding=1),
            nn.InstanceNorm3d(out_ch),
            nn.LeakyReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)


In [ ]:
class CBCTEncoder(nn.Module):
    def __init__(self, in_channels=1, base_ch=32, emb_dim=512):
        super().__init__()
        # Deeper architecture with gradual channel increase
        self.enc1 = ConvBlock3D(in_channels, base_ch, dropout=0.1)
        self.enc2 = ConvBlock3D(base_ch, base_ch * 2, dropout=0.15)
        self.enc3 = ConvBlock3D(base_ch * 2, base_ch * 4, dropout=0.2)
        self.enc4 = ConvBlock3D(base_ch * 4, base_ch * 8, dropout=0.25)
        self.enc5 = ConvBlock3D(base_ch * 8, base_ch * 16, dropout=0.3)

        self.pool = nn.MaxPool3d(2)
        self.global_pool = nn.AdaptiveAvgPool3d(1)

        # Embedding layer
        self.embedding = nn.Sequential(
            nn.Linear(base_ch * 16, emb_dim),
            nn.LayerNorm(emb_dim),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3)
        )

    def forward(self, x):
        # Encoder pathway with skip connections stored
        x1 = self.enc1(x)       # 224 -> 224
        x = self.pool(x1)       # 224 -> 112

        x2 = self.enc2(x)       # 112 -> 112
        x = self.pool(x2)       # 112 -> 56

        x3 = self.enc3(x)       # 56 -> 56
        x = self.pool(x3)       # 56 -> 28

        x4 = self.enc4(x)       # 28 -> 28
        x = self.pool(x4)       # 28 -> 14

        x5 = self.enc5(x)       # 14 -> 14

        # Global pooling and embedding
        feat = self.global_pool(x5).view(x5.size(0), -1)
        feat = self.embedding(feat)

        return feat

In [ ]:
class CBCTClassifier(nn.Module):
    def __init__(self, num_classes, emb_dim=512, base_ch=32):
        super().__init__()
        self.encoder = CBCTEncoder(
            in_channels=1,
            base_ch=base_ch,
            emb_dim=emb_dim
        )

        # More robust classifier head
        self.classifier = nn.Sequential(
            nn.Linear(emb_dim, emb_dim // 2),
            nn.LayerNorm(emb_dim // 2),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.4),
            nn.Linear(emb_dim // 2, num_classes)
        )

    def forward(self, x):
        features = self.encoder(x)
        logits = self.classifier(features)
        return logits, features


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model = CBCTClassifier(
    num_classes=len(DISEASE_CLASSES),
    emb_dim=512,
    base_ch=32
).cuda()

criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))

# optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4,
    betas=(0.9, 0.999)
)

# Learning rate scheduler
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='max',
    factor=0.5,
    patience=7,
    verbose=True,
    min_lr=1e-7
)


c:\Users\T2510590\.conda\envs\tf_cuda11_8\lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


In [ ]:
from sklearn.metrics import accuracy_score, classification_report


In [ ]:
def evaluate_accuracy(model, loader):
    model.eval()
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            logits, features = model(x)

            preds = torch.argmax(logits, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_targets.extend(y.numpy())

    acc = accuracy_score(all_targets, all_preds)
    return acc, all_targets, all_preds

In [ ]:
import gc
gc.collect()
torch.cuda.empty_cache()


In [ ]:
EPOCHS = 100
patience = 5
best_val_acc = 0.0
epochs_no_improve = 0

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for x, y in train_loader:

        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()

        logits, features = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()


    avg_loss = total_loss / len(train_loader)

    val_acc, _, _ = evaluate_accuracy(model, val_loader)

    # ---------- EARLY STOPPING ----------
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        epochs_no_improve = 0

        # save best model
        torch.save(model.state_dict(), "best_image_model.pth")
    else:
        epochs_no_improve += 1

    print(
        f"Epoch {epoch+1} | "
        f"Loss: {avg_loss:.4f} | "
        f"Val Accuracy: {val_acc:.4f} | "
    )

    if epochs_no_improve >= patience:
        break


Epoch 1 | Loss: 2.1316 | Val Accuracy: 0.1443 | 
Epoch 2 | Loss: 2.1061 | Val Accuracy: 0.1443 | 
Epoch 3 | Loss: 2.0980 | Val Accuracy: 0.1649 | 
Epoch 4 | Loss: 2.0509 | Val Accuracy: 0.2062 | 
Epoch 5 | Loss: 2.0520 | Val Accuracy: 0.2062 | 
Epoch 6 | Loss: 2.0429 | Val Accuracy: 0.2062 | 
Epoch 7 | Loss: 2.0498 | Val Accuracy: 0.2062 | 
Epoch 8 | Loss: 2.0285 | Val Accuracy: 0.2062 | 
Epoch 9 | Loss: 2.0732 | Val Accuracy: 0.1649 | 


In [ ]:

_, y_true, y_pred = evaluate_accuracy(model, val_loader)

print(
    classification_report(
        y_true,
        y_pred,
        target_names=DISEASE_CLASSES
    )
)

                          precision    recall  f1-score   support

    Apical periodontitis       0.00      0.00      0.00        14
           Periodontitis       0.00      0.00      0.00         8
              Gingivitis       0.00      0.00      0.00        14
           Crooked teeth       0.00      0.00      0.00         2
Damaged or missing teeth       0.16      1.00      0.28        16
          Impacted tooth       0.00      0.00      0.00        20
                  Caries       0.00      0.00      0.00        11
                Pulpitis       0.00      0.00      0.00        12

                accuracy                           0.16        97
               macro avg       0.02      0.12      0.04        97
            weighted avg       0.03      0.16      0.05        97



c:\Users\T2510590\.conda\envs\tf_cuda11_8\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\T2510590\.conda\envs\tf_cuda11_8\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\T2510590\.conda\envs\tf_cuda11_8\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", le